In [1]:
!pip install torch torchvision torchaudio scikit-learn tqdm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import pandas as pd
import numpy as np
import torch

ROOT        = "/content/drive/MyDrive/Hackenza_MUPlovers"
FEATURE_DIR = os.path.join(ROOT, "cache/features_normalized")  # ← normalized features
TRAIN_MANIFEST = os.path.join(ROOT, "data/train_manifest_split.csv")
VAL_MANIFEST   = os.path.join(ROOT, "data/val_manifest_split.csv")

print("Feature dir exists:", os.path.exists(FEATURE_DIR))
print("Train manifest exists:", os.path.exists(TRAIN_MANIFEST))
print("Val manifest exists:", os.path.exists(VAL_MANIFEST))

Feature dir exists: True
Train manifest exists: True
Val manifest exists: True


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [5]:
from torch.utils.data import Dataset

class NativeDataset(Dataset):
    def __init__(self, manifest_csv, feature_dir):
        self.df = pd.read_csv(manifest_csv)
        self.feature_dir = feature_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_id = str(row["file_id"])
        label   = float(row["label"])

        feature_path = os.path.join(self.feature_dir, f"{file_id}.npy")
        x = np.load(feature_path)
        x = torch.tensor(x, dtype=torch.float32)

        return x, torch.tensor(label, dtype=torch.float32)

In [6]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    sequences, labels = zip(*batch)

    lengths = [seq.shape[0] for seq in sequences]
    padded  = pad_sequence(sequences, batch_first=True)  # [B, T_max, 783]

    mask = torch.zeros(padded.shape[:2])
    for i, length in enumerate(lengths):
        mask[i, :length] = 1

    labels = torch.stack(labels)

    return padded, mask, labels

In [7]:
from torch.utils.data import DataLoader

train_dataset = NativeDataset(TRAIN_MANIFEST, FEATURE_DIR)
val_dataset   = NativeDataset(VAL_MANIFEST,   FEATURE_DIR)

train_loader = DataLoader(
    train_dataset,
    batch_size  = 8,
    shuffle     = True,
    collate_fn  = collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size  = 8,
    shuffle     = False,
    collate_fn  = collate_fn
)

print("Train size:", len(train_dataset))
print("Val size  :", len(val_dataset))

Train size: 112
Val size  : 24


In [8]:
# Automatically compute class weight from training data
train_df = pd.read_csv(TRAIN_MANIFEST)
num_native     = (train_df['label'] == 1).sum()
num_non_native = (train_df['label'] == 0).sum()

pos_weight = torch.tensor([num_non_native / num_native], dtype=torch.float32).to(device)

print(f"Native samples    : {num_native}")
print(f"Non-native samples: {num_non_native}")
print(f"pos_weight        : {pos_weight.item():.4f}")

Native samples    : 80
Non-native samples: 32
pos_weight        : 0.4000


In [9]:
import torch.nn as nn
import torch.nn.functional as F

class TemporalEncoder(nn.Module):
    def __init__(self, input_dim=783, hidden_dim=64):  # ← reduced from 256 to 64
        super().__init__()
        self.gru = nn.GRU(
            input_dim,
            hidden_dim,
            batch_first  = True,
            bidirectional= True
        )
        self.output_dim = hidden_dim * 2

    def forward(self, x):
        out, _ = self.gru(x)
        return out


class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x, mask):
        scores  = self.attn(x).squeeze(-1)
        scores  = scores.masked_fill(mask == 0, -1e9)
        weights = torch.softmax(scores, dim=1)
        pooled  = torch.sum(x * weights.unsqueeze(-1), dim=1)
        return pooled


class AccentModel(nn.Module):
    def __init__(self, input_dim=783):
        super().__init__()
        self.encoder    = TemporalEncoder(input_dim)
        self.pool       = AttentionPooling(self.encoder.output_dim)
        self.classifier = nn.Sequential(
            nn.Linear(self.encoder.output_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x, mask):
        seq    = self.encoder(x)
        pooled = self.pool(seq, mask)
        logits = self.classifier(pooled).squeeze(-1)
        return logits

In [13]:
import torch.optim as optim

model     = AccentModel(783).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)  # ← reduced from 1e-3
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    patience = 3,
    factor   = 0.5,
)

print("✅ Model ready!")
print("Total parameters:", sum(p.numel() for p in model.parameters()))

✅ Model ready!
Total parameters: 342786


In [15]:
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

def train_epoch():
    model.train()
    total_loss = 0

    for x, mask, y in tqdm(train_loader):
        x, mask, y = x.to(device), mask.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x, mask)

        # ← pos_weight added here
        loss = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(train_loader)


def evaluate():
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0

    with torch.no_grad():
        for x, mask, y in val_loader:
            x, mask, y = x.to(device), mask.to(device), y.to(device)

            logits = model(x, mask)
            loss   = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pos_weight)
            total_loss += loss.item()

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    acc     = accuracy_score(all_labels, all_preds)
    f1      = f1_score(all_labels, all_preds, zero_division=0)
    val_loss = total_loss / len(val_loader)

    return acc, f1, val_loss

In [18]:
EPOCHS   = 50
best_acc = 0
best_f1  = 0

for epoch in range(EPOCHS):
    loss              = train_epoch()
    acc, f1, val_loss = evaluate()

    # Step scheduler based on val loss
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss : {loss:.4f}")
    print(f"  Val Loss   : {val_loss:.4f}")
    print(f"  Val Acc    : {acc:.4f}")
    print(f"  Val F1     : {f1:.4f}")

    # Save best model
    if acc > best_acc:
        best_acc = acc
        best_f1  = f1
        os.makedirs('/content/drive/MyDrive/Hackenza_MUPlovers/runs/', exist_ok=True)
        torch.save(model.state_dict(), '/content/drive/MyDrive/Hackenza_MUPlovers/runs/best_model.pt')
        print(f"  ✅ Best model saved!")

    print("-" * 40)

print(f"\n🏆 Best Accuracy: {best_acc:.4f}")
print(f"🏆 Best F1      : {best_f1:.4f}")

100%|██████████| 14/14 [00:05<00:00,  2.74it/s]


Epoch 1/50
  Train Loss : 0.3848
  Val Loss   : 0.3939
  Val Acc    : 0.7917
  Val F1     : 0.8649
  ✅ Best model saved!
----------------------------------------


100%|██████████| 14/14 [00:02<00:00,  6.18it/s]


Epoch 2/50
  Train Loss : 0.3775
  Val Loss   : 0.3890
  Val Acc    : 0.7917
  Val F1     : 0.8649
----------------------------------------


100%|██████████| 14/14 [00:01<00:00,  8.55it/s]


...

Epoch 48/50
  Train Loss : 0.0405
  Val Loss   : 0.2843
  Val Acc    : 0.9167
  Val F1     : 0.9444
----------------------------------------


100%|██████████| 14/14 [00:01<00:00,  9.61it/s]


Epoch 49/50
  Train Loss : 0.0401
  Val Loss   : 0.2844
  Val Acc    : 0.9167
  Val F1     : 0.9444
----------------------------------------


100%|██████████| 14/14 [00:01<00:00,  9.69it/s]

Epoch 50/50
  Train Loss : 0.0415
  Val Loss   : 0.2844
  Val Acc    : 0.9167
  Val F1     : 0.9444
----------------------------------------

🏆 Best Accuracy: 0.9167
🏆 Best F1      : 0.9444
